<!-- 🧠 Prompt: "armá la portada del notebook con título, alumno, fuente elegida, objetivo y el diagrama del pipeline" -->

# TP2 — Web scraping y preprocesamiento
### Procesamiento del Habla (PH) · ISSD · 4° IAR

**Alumno:** Nicolás Bargioni

**Página elegida:** *"Del cuento breve y sus alrededores"* — **Julio Cortázar** (conferencia/ensayo), publicada en **Ciudad Seva**.
URL: https://ciudadseva.com/texto/del-cuento-breve-y-sus-alrededores/

> Sitio **estático** en español, texto largo (~3600 palabras) y limpio en `<article>`. Ningún compañero usó esta fuente.

## Objetivo
Aplicar la **etapa 1 del pipeline de PH** (adquisición → limpieza) pero esta vez con datos provenientes de la **web**: extraer el texto con `requests` + `BeautifulSoup` (la *"sopa de letras"* vista en clase) y preprocesarlo (tokenización, stopwords, stemming/lematización) para un análisis de frecuencias.

```
Web (HTML) → requests → BeautifulSoup → texto crudo → limpieza/stopwords → tokens → frecuencias
```

<!-- 🧠 Prompt: "escribí una intro corta para la sección de instalación de dependencias" -->

## 0. Preparación del entorno

In [ ]:
# 🧠 Prompt: "instalá las dependencias necesarias en Colab (requests, pymupdf, pdfplumber, nltk, spacy)"
!pip -q install requests beautifulsoup4 html5lib nltk spacy
!python -m spacy download es_core_news_sm -q


In [ ]:
# 🧠 Prompt: "importá las librerías y descargá los recursos de NLTK y el modelo de spaCy"
import requests
from bs4 import BeautifulSoup        # "sopa de letras" (clase 2)
import re, string
from collections import Counter
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
for rec in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(rec, quiet=True)

import spacy
nlp = spacy.load("es_core_news_sm")
print("Entorno listo ✔")


<!-- 🧠 Prompt: "explicá en markdown el paso de adquisición y por qué se usa un User-Agent para evitar el 403" -->

## 1. Adquisición — descarga del HTML

Usamos `requests` con un **User-Agent de navegador**. En clase vimos que muchos servidores devuelven **403 Forbidden** ante un script "pelado"; simular Chrome con headers lo evita.

In [ ]:
# 🧠 Prompt: "descargá el archivo/página con requests usando headers de navegador y mostrá el status code"
URL = "https://ciudadseva.com/texto/del-cuento-breve-y-sus-alrededores/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/124.0 Safari/537.36"}

respuesta = requests.get(URL, headers=headers, timeout=30)
print("Status code:", respuesta.status_code)      # 200 = OK ; 403 = bloqueado
print("Content-Type:", respuesta.headers.get("Content-Type"))
print("Tamaño del HTML:", len(respuesta.content), "bytes")


<!-- 🧠 Prompt: "introducí el parseo del HTML con BeautifulSoup (la sopa de letras)" -->

### 1.1 Armamos la "sopa de letras"
Parseamos el HTML con `BeautifulSoup` usando el parser **`html5lib`** (el que usó la profe en el notebook de clase; es tolerante a HTML mal formado).

In [ ]:
# 🧠 Prompt: "parseá el HTML con BeautifulSoup usando el parser html5lib"
sopa = BeautifulSoup(respuesta.content, "html5lib")
print("Título de la pestaña (<title>):", sopa.find("title").get_text(strip=True))


<!-- 🧠 Prompt: "explicá cómo explorar la estructura HTML (h1, autor, párrafos) antes de extraer" -->

## 2. Exploración de la estructura HTML

Antes de extraer texto, miramos **cómo está organizada la página**: el título principal (`h1`), el autor y los párrafos del cuerpo.

In [ ]:
# 🧠 Prompt: "extraé el título (h1) y derivá el autor desde el <title>"
# Título principal del documento
h1 = sopa.find("h1").get_text(strip=True)
print("h1 (título):", h1)

# Autor: lo derivamos del <title> ("... - Julio Cortázar - Ciudad Seva ...")
titulo_tag = sopa.find("title").get_text(strip=True)
autor = titulo_tag.split(" - ")[1] if " - " in titulo_tag else "(no detectado)"
print("Autor:", autor)

# ¿Dónde está el cuerpo del texto? Probamos contenedores
art = sopa.find("article")
print("Párrafos <p> dentro de <article>:", len(art.find_all("p")))


<!-- 🧠 Prompt: "compará get_text() crudo contra la extracción dirigida por <article>/<p>" -->

**Observación.** El texto vive dentro de `<article>` (≈14 `<p>`). Extraer por `<article>`/`<p>` es **mucho más limpio** que `get_text()` sobre toda la página, que arrastra menú, pie y navegación. Lo comparamos abajo.

<!-- 🧠 Prompt: "explicá el análisis estructural del texto crudo y los caracteres separadores" -->

## 3. Extracción del texto — crudo vs. dirigido

### 3.1 `get_text()` sobre toda la sopa (lo más crudo)
La función `sopa.get_text()` devuelve **todo** el texto del HTML, incluyendo ruido (menú, links, footer).

In [ ]:
# 🧠 Prompt: "obtené el texto crudo de toda la página con get_text() y mostrá una muestra"
texto_crudo = sopa.get_text()
print("Longitud texto crudo (toda la página):", len(texto_crudo), "chars")
print(repr(texto_crudo[:300]))     # repr() para ver \n y espacios


<!-- 🧠 Prompt: "compará get_text() crudo contra la extracción dirigida por <article>/<p>" -->

### 3.2 Extracción dirigida (solo el cuerpo del artículo)
Tomamos únicamente los `<p>` del `<article>` y los unimos. Mucho menos ruido.

In [ ]:
# 🧠 Prompt: "extraé los párrafos del <article> y armá el texto limpio del cuerpo"
parrafos = [p.get_text(" ", strip=True) for p in art.find_all("p")]
parrafos = [p for p in parrafos if p]              # descartar vacíos
texto = "\n\n".join(parrafos)

print("Párrafos extraídos:", len(parrafos))
print("Longitud texto del artículo:", len(texto), "chars")
print("\n----- primeros 2 párrafos -----")
for p in parrafos[:2]:
    print("•", p[:160], "...\n")


<!-- 🧠 Prompt: "explicá cómo se quitan stopwords y puntuación antes de contar frecuencias" -->

**Detalle real (y punto de análisis).** El ensayo **abre con un epígrafe en francés** (*"León L. affirmait qu'il n'y avait qu'une chose..."*). Es un **texto mixto** español + francés: lo dejamos a propósito porque más adelante mostrará que **la lista de stopwords depende del idioma** (las palabras francesas NO se filtran con stopwords en español).

<!-- 🧠 Prompt: "explicá la tokenización en español y por qué declarar el idioma" -->

## 4. Preprocesamiento (etapa 1 del pipeline)

### 4.1 Tokenización + minúsculas
Tokenizamos **declarando idioma español** (en clase se remarcó: tokenizar en español ≠ en inglés).

In [ ]:
# 🧠 Prompt: "tokenizá el texto en minúsculas declarando idioma español"
texto_min = texto.lower()
tokens = word_tokenize(texto_min, language="spanish")
print("Total de tokens (crudos):", len(tokens))
print("Muestra:", tokens[:20])


<!-- 🧠 Prompt: "explicá cómo se quitan stopwords y puntuación antes de contar frecuencias" -->

### 4.2 Limpieza: stopwords + puntuación
Quitamos stopwords del español (NLTK) y puntuación. Descartamos también tokens muy cortos y los que son sólo números/símbolos.

In [ ]:
# 🧠 Prompt: "filtrá stopwords del español, puntuación, números y tokens cortos"
stop_es = set(stopwords.words("spanish"))
puntuacion = set(string.punctuation) | {"…","”","“","«","»","—","–","‘","’","¿","¡","º"}

def es_palabra_util(tok):
    if tok in stop_es:    return False
    if tok in puntuacion: return False
    if len(tok) <= 2:     return False
    if re.fullmatch(r"[\d.,;:%/()\-–—'’]+", tok): return False
    return True

tokens_limpios = [t for t in tokens if es_palabra_util(t)]
print("Tokens tras limpieza:", len(tokens_limpios))
print("Muestra:", tokens_limpios[:25])


<!-- 🧠 Prompt: "introducí el análisis de frecuencias y la limpieza con NLTK" -->

## 5. Análisis de frecuencias

In [ ]:
# 🧠 Prompt: "calculá las 15 palabras más frecuentes con Counter"
frec = Counter(tokens_limpios)
top15 = frec.most_common(15)
print(top15)

palabras = [p for p,_ in top15]
valores  = [c for _,c in top15]

plt.figure(figsize=(10,5))
plt.barh(palabras[::-1], valores[::-1])
plt.title("Top 15 palabras más frecuentes — Cortázar, 'Del cuento breve y sus alrededores'")
plt.xlabel("Frecuencia")
plt.tight_layout()
plt.show()


<!-- 🧠 Prompt: "explicá cómo se quitan stopwords y puntuación antes de contar frecuencias" -->

### 5.1 ¿Aparecen tokens que también considerarías *stopwords*? — análisis

**Sí**, en dos sentidos:

- **Stopwords de dominio:** como el texto es un ensayo *sobre el cuento*, palabras como **"cuento", "cuentista", "cuentos"** dominan el ranking: son el tema, pero **no discriminan** entre párrafos. En un análisis temático conviene moverlas a una **lista de stopwords del dominio** (igual que en clase: en un discurso político *"gobierno"* sería stopword de dominio).
- **Stopwords de OTRO idioma (hallazgo del texto mixto):** por el **epígrafe en francés**, se cuelan tokens como *"chose", "journée", "cadre"* (palabras francesas de bajo contenido) que la lista de NLTK en **español** no conoce y por eso no filtra. Esto demuestra de forma muy concreta que **las stopwords dependen del idioma**: para limpiar bien un corpus multilingüe habría que detectar idioma y combinar listas.

**Conclusión.** La limpieza no es universal: depende del **idioma** y del **dominio**. Una segunda pasada sumando stopwords de dominio (`cuento`, `cuentos`, `cuentista`) y del francés dejaría ver mejor los conceptos propios del ensayo (p. ej. *tensión, escritura, lector, tiempo*).

<!-- 🧠 Prompt: "compará stemming (NLTK) vs lematización (spaCy) como recurso de clase" -->

### 5.2 (Extra) Stemming vs. Lematización — recurso de clase

In [ ]:
# 🧠 Prompt: "compará stemming de NLTK con la lematización de spaCy sobre las palabras top"
stemmer = SnowballStemmer("spanish")
muestra = palabras[:8]
doc = nlp(" ".join(muestra))
lemas = {t.text: t.lemma_ for t in doc}

print(f"{'palabra':15}{'stemming (NLTK)':22}{'lema (spaCy)':18}")
print("-"*55)
for w in muestra:
    print(f"{w:15}{stemmer.stem(w):22}{lemas.get(w,w):18}")


<!-- 🧠 Prompt: "compará stemming (NLTK) vs lematización (spaCy) como recurso de clase" -->

**Lectura.** El *stemming* recorta a la raíz (ej. *cuentista → cuent*), agrupando `cuento/cuentos/cuentista` bajo una misma forma — útil para reducir el vocabulario. La *lematización* devuelve la forma de diccionario (más interpretable). En clase se vio que la elección depende del objetivo (vectorizar vs. interpretar).

<!-- 🧠 Prompt: "introducí el parseo del HTML con BeautifulSoup (la sopa de letras)" -->

## 6. Nota sobre sitios dinámicos (Selenium)

La página elegida es **estática**: todo el texto viene en el HTML inicial, por eso `requests` + `BeautifulSoup` alcanzan. Para sitios **dinámicos** (contenido cargado por JavaScript), el HTML inicial llega vacío y hay que usar **Selenium** (navegador real), como mostró la profe:

```python
from selenium import webdriver
driver = webdriver.Chrome()
driver.get(url)
html = driver.page_source          # HTML ya renderizado por el navegador
soup = BeautifulSoup(html, "html5lib")
driver.quit()
```

No fue necesario acá porque el contenido ya estaba en el HTML servido (lo confirmamos: `requests` trajo el texto completo).

<!-- 🧠 Prompt: "explicá en markdown el paso de adquisición y por qué se usa un User-Agent para evitar el 403" -->

## 7. Conclusiones

- **Adquisición web:** `requests` + User-Agent descargó el HTML sin bloqueo (status 200) y `BeautifulSoup` (`html5lib`) lo parseó.
- **Estructura:** el cuerpo vive en `<article>`/`<p>`; **extraer dirigido** (`find_all("p")`) es mucho más limpio que `get_text()` sobre toda la página (que arrastra menú/footer).
- **Paso más desafiante:** distinguir el **texto útil del ruido** del HTML y manejar el **texto mixto** (epígrafe en francés).
- **Limpieza:** tras quitar stopwords + puntuación, el ranking quedó dominado por **stopwords de dominio** (*cuento, cuentos, cuentista*) y se colaron **stopwords francesas** del epígrafe → confirma que las stopwords dependen del **idioma** y del **dominio**.
- Se recorrió la **etapa 1 del pipeline** desde una fuente **web** (complemento del TP1, que partía de un PDF).

## 8. Referencias
- **Fuente:** Cortázar, J. *Del cuento breve y sus alrededores*. Ciudad Seva. https://ciudadseva.com/texto/del-cuento-breve-y-sus-alrededores/
- **Notebook de clase (template):** https://github.com/anadiedrichs/procesamientoDelHabla/blob/main/clase_2_webscrapping.ipynb
- **requests:** https://requests.readthedocs.io/ · **BeautifulSoup:** https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- **NLTK:** https://www.nltk.org/ · **spaCy (es_core_news_sm):** https://spacy.io/models/es
- **Asistencia IA:** _[completar: pegar acá el enlace a la conversación con la IA utilizada]_